In [1]:
from huggingface_hub import login, HfApi
import pandas as pd
import nltk
import numpy as np
from datasets import Dataset
import evaluate
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType
import torch

In [2]:
nltk.download('punkt', quiet=True)

# Load your CSV file
csv_path = "data.csv" 
df = pd.read_csv(csv_path)

# Ensure the CSV has 'question' and 'answer' columns
assert 'Question' in df.columns and 'Answers' in df.columns, "CSV must contain 'Question' and 'Answers' columns"

# Convert to Hugging Face Dataset and split
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2)

In [3]:


# Load tokenizer and quantized model
tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-small").to("cpu")


# Prepare PEFT/LoRA configuration for efficient fine-tuning
peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    inference_mode=False,
    r=4,  # LoRA rank, smaller for CPU
    lora_alpha=16,  # LoRA alpha
    lora_dropout=0.05,
    target_modules=["q", "v"]  # Target modules for T5
)

#model = get_peft_model(model, peft_config)
#model.print_trainable_parameters()

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [4]:
prefix = "answer the question: "
def preprocess_function(examples):
    inputs = [prefix + doc for doc in examples["Question"]]
    model_inputs = tokenizer(inputs, max_length=64, truncation=True)  # Shorter for CPU
    labels = tokenizer(text_target=examples["Answers"], max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs
tokenized_dataset = dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/136 [00:00<?, ? examples/s]

Map:   0%|          | 0/35 [00:00<?, ? examples/s]

In [6]:
# 6. Training setup (CPU-optimized)
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    learning_rate=3e-4,  # Lower for CPU
    per_device_train_batch_size=8, 
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    num_train_epochs=30, 
    save_total_limit=3,
    save_strategy="steps",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model),
    compute_metrics=lambda eval_pred: {
        k: round(v * 100, 4) 
        for k, v in evaluate.load("rouge").compute(
            predictions=["\n".join(nltk.sent_tokenize(pred.strip())) 
            for pred in tokenizer.batch_decode(eval_pred.predictions, skip_special_tokens=True)],
            references=["\n".join(nltk.sent_tokenize(label.strip())) 
            for label in tokenizer.batch_decode(
                np.where(eval_pred.label_ids != -100, eval_pred.label_ids, tokenizer.pad_token_id), 
                skip_special_tokens=True)],
            use_stemmer=True
        ).items()
    }
)

# 7. Train
print("Starting training (may be slow on CPU)...")
trainer.train()
print("Training complete!")


/var/folders/4t/1kd6_nl521zg34vxn7hq4k0m0000gn/T/ipykernel_56743/1260422178.py:13: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Starting training (may be slow on CPU)...


Step,Training Loss
500,1.339100


Training complete!


In [7]:
from huggingface_hub import notebook_login

notebook_login()

In [8]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
model = model.to(device)
print(f"Using device: {device}")

# 1. Perform test inference
print("\n=== Testing the trained model ===")
test_question = "What are Bollinger Bands?"
input_text = "answer the question: " + test_question

# Tokenize and generate
inputs = tokenizer(input_text, max_length=64, return_tensors="pt").to(device)
outputs = model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_length=50
)
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"Question: {test_question}")
print(f"Generated Answer: {answer}\n")


# 2. Push to Hub
model = model.to("cpu")
hub_model_id = "edbertw/tuned_flanT5" 
model.push_to_hub(hub_model_id)
tokenizer.push_to_hub(hub_model_id)

print(f"\nModel successfully pushed to: https://huggingface.co/{hub_model_id}")
print("You can now use it with:")
print(f"from transformers import AutoModelForSeq2SeqLM, AutoTokenizer")
print(f"model = AutoModelForSeq2SeqLM.from_pretrained('{hub_model_id}')")
print(f"tokenizer = AutoTokenizer.from_pretrained('{hub_model_id}')")

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


Using device: mps

=== Testing the trained model ===
Question: What are Bollinger Bands?
Generated Answer: The two most common bands of the same price, ranging from 20-day moving averages to 50-day moving averages.



model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.



Model successfully pushed to: https://huggingface.co/edbertw/tuned_flanT5
You can now use it with:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
model = AutoModelForSeq2SeqLM.from_pretrained('edbertw/tuned_flanT5')
tokenizer = AutoTokenizer.from_pretrained('edbertw/tuned_flanT5')
